# Laboratorio de espectros: líneas rectas y su transformada de Fourier

**Procesamiento de Imágenes — Universidad Diego Portales**

Este cuaderno tiene un *pool* de imágenes binarias sintéticas, ordenadas de menor a
mayor complejidad (una línea, líneas perpendiculares, diagonales, familias de líneas
paralelas, rejillas, etc.). Se elige una imagen de entrada y se obtiene su espectro de
magnitud 2D para analizar sus frecuencias.

**Idea clave para leer los espectros:** una estructura lineal en la imagen concentra su
energía en una línea **perpendicular** a ella en el espectro, que pasa por el origen.
Si la estructura se repite con período $P$, el espectro se vuelve una serie de puntos
separados por $1/P$ ciclos/píxel.

## 1. Preparación y parámetros

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Parámetros generales ----
TAM = 256        # las imágenes son TAM x TAM píxeles
GROSOR = 1       # grosor de las líneas (px)
PERIODO = 16     # separación entre líneas en los patrones periódicos (px)

plt.rcParams["image.cmap"] = "gray"

## 2. Pool de imágenes binarias

Todas las líneas se dibujan con la misma idea: con coordenadas centradas $(x, y)$,
una recta que pasa por el origen con ángulo $\theta$ (medido desde el eje horizontal)
es el conjunto de puntos a distancia menor que `GROSOR/2` de ella:

$$\left| -x\sin\theta + y\cos\theta \right| < \tfrac{\text{GROSOR}}{2}$$

Las **familias de rectas paralelas** se definen con una dirección entera $(a, b)$:

$$(a\,x + b\,y) \bmod P < \text{GROSOR}$$

- $(1, 0)$: verticales, $(0, 1)$: horizontales, $(1, 1)$ y $(1, -1)$: diagonales a 45°, $(1, 2)$: pendiente 1:2 (≈ 26.6°).
- Con $a, b$ enteros y $P$ divisor de `TAM`, el patrón empalma exactamente con su copia periódica,
  así que su espectro son **puntos limpios** en $(u, v) = k\,(a, b)/P$.
Con esta convención, $\theta = 0°$ es horizontal y $\theta = 90°$ es vertical; el eje $y$
de la imagen crece hacia abajo, por lo que un ángulo positivo se ve inclinado hacia abajo
a la derecha.

In [ ]:
def coordenadas(tam):
    # Coordenadas enteras centradas: el píxel (tam//2, tam//2) es el origen
    y, x = np.mgrid[0:tam, 0:tam]
    return x - tam // 2, y - tam // 2


def recta(tam, angulo, grosor=1, desplazamiento=0):
    # Recta con ángulo `angulo` (grados) desplazada `desplazamiento` px de su normal
    x, y = coordenadas(tam)
    t = np.deg2rad(angulo)
    d = -x * np.sin(t) + y * np.cos(t) - desplazamiento
    return np.abs(d) < grosor / 2


def rectas_paralelas(tam, a, b, periodo, grosor=1):
    # Familia de rectas a*x + b*y = k*periodo, con (a, b) enteros
    x, y = coordenadas(tam)
    return np.mod(a * x + b * y, periodo) < grosor


def a_binaria(mascara):
    return mascara.astype(np.float64)

In [ ]:
def construir_pool(tam=TAM, g=GROSOR, p=PERIODO):
    x, y = coordenadas(tam)
    pool = {}

    # Nivel 1: una recta
    pool["01_vertical"]            = recta(tam, 90, g)
    pool["02_horizontal"]          = recta(tam, 0, g)
    pool["03_vertical_desplazada"] = recta(tam, 90, g, desplazamiento=-tam // 4)

    # Nivel 2: rectas perpendiculares
    pool["04_cruz"]                = recta(tam, 0, g) | recta(tam, 90, g)

    # Nivel 3: diagonales
    pool["05_diagonal_45"]         = recta(tam, 45, g)
    pool["06_diagonal_135"]        = recta(tam, 135, g)
    pool["07_equis"]               = recta(tam, 45, g) | recta(tam, 135, g)
    pool["08_recta_30"]            = recta(tam, 30, g)
    pool["09_recta_60"]            = recta(tam, 60, g)

    # Nivel 4: varias rectas por el origen
    pool["10_asterisco"]           = recta(tam, 0, g) | recta(tam, 45, g) | recta(tam, 90, g) | recta(tam, 135, g)
    pool["11_estrella_12"]         = np.any([recta(tam, a, g) for a in range(0, 180, 15)], axis=0)

    # Nivel 5: familias de rectas paralelas (patrones periódicos)
    pool["12_verticales"]          = rectas_paralelas(tam, 1, 0, p, g)
    pool["13_horizontales"]        = rectas_paralelas(tam, 0, 1, p, g)
    pool["14_verticales_densas"]   = rectas_paralelas(tam, 1, 0, p // 2, g)
    pool["15_diagonales_45"]       = rectas_paralelas(tam, 1, 1, p, g)
    pool["16_diagonales_1a2"]      = rectas_paralelas(tam, 1, 2, p, g)

    # Nivel 6: combinaciones de familias
    pool["17_rejilla"]             = rectas_paralelas(tam, 1, 0, p, g) | rectas_paralelas(tam, 0, 1, p, g)
    pool["18_rejilla_diagonal"]    = rectas_paralelas(tam, 1, 1, p, g) | rectas_paralelas(tam, 1, -1, p, g)
    pool["19_rejilla_mixta"]       = rectas_paralelas(tam, 1, 0, p, g) | rectas_paralelas(tam, 0, 1, 2 * p, g)

    # Nivel 7: formas cerradas hechas de rectas (y una curva para contrastar)
    a = tam // 4
    pool["20_cuadrado_contorno"]   = (np.maximum(np.abs(x), np.abs(y)) == a)
    pool["21_cuadrado_relleno"]    = (np.maximum(np.abs(x), np.abs(y)) <= a)
    pool["22_rombo_contorno"]      = (np.abs(np.abs(x) + np.abs(y) - a) < g / 2 + 0.5)
    pool["23_circulo_contorno"]    = (np.abs(np.hypot(x, y) - a) < g / 2 + 0.5)

    return {nombre: a_binaria(m) for nombre, m in pool.items()}


POOL = construir_pool()
print(f"{len(POOL)} imágenes disponibles:")
for nombre in POOL:
    print("  -", nombre)

Vista rápida de todo el pool:

In [ ]:
def mostrar_pool(pool, cols=6):
    nombres = list(pool)
    filas = int(np.ceil(len(nombres) / cols))
    fig, ax = plt.subplots(filas, cols, figsize=(2.6 * cols, 2.8 * filas))
    for a in ax.ravel():
        a.axis("off")
    for a, nombre in zip(ax.ravel(), nombres):
        a.imshow(pool[nombre], vmin=0, vmax=1, interpolation="antialiased")
        a.set_title(nombre, fontsize=9)
    plt.tight_layout()
    plt.show()


mostrar_pool(POOL)

## 3. Espectro de una imagen

`espectro(img)` devuelve $\log(1+|F(u,v)|)$ centrado con `fftshift`, y los ejes de
frecuencia en ciclos/píxel (de $-0.5$ a $0.5$).

`analizar(img)` muestra la imagen, el espectro completo y una **vista ampliada** alrededor
del origen (útil para ver los picos de los patrones periódicos, que están cerca del centro).

Opción `ventana=True`: multiplica la imagen por una ventana de Hann antes de la FFT.
La DFT supone que la imagen se repite periódicamente; si el contenido no "empalma" con su
copia en el borde opuesto, aparece energía espuria repartida por el espectro (*leakage*).
La ventana atenúa los bordes y reduce ese efecto.

In [ ]:
def espectro(img, ventana=False):
    f = img.astype(np.float64)
    if ventana:
        f = f * np.outer(np.hanning(f.shape[0]), np.hanning(f.shape[1]))
    F = np.fft.fftshift(np.fft.fft2(f))
    u = np.fft.fftshift(np.fft.fftfreq(f.shape[1]))
    v = np.fft.fftshift(np.fft.fftfreq(f.shape[0]))
    return np.log1p(np.abs(F)), u, v


def analizar(img, titulo="", ventana=False, zoom=0.125, cmap="inferno"):
    # zoom: semiancho de la vista ampliada, en ciclos/píxel (máximo 0.5)
    S, u, v = espectro(img, ventana)
    ext = [u[0], u[-1], v[-1], v[0]]

    fig, ax = plt.subplots(1, 3, figsize=(17, 5.2))

    ax[0].imshow(img, vmin=0, vmax=1, interpolation="nearest")
    ax[0].set_title(f"Imagen: {titulo}")
    ax[0].axis("off")

    im = ax[1].imshow(S, extent=ext, cmap=cmap, interpolation="nearest")
    ax[1].set_title(r"Espectro $\log(1+|F(u,v)|)$" + ("  [Hann]" if ventana else ""))
    fig.colorbar(im, ax=ax[1], fraction=0.046)

    ax[2].imshow(S, extent=ext, cmap=cmap, interpolation="nearest")
    ax[2].set_xlim(-zoom, zoom)
    ax[2].set_ylim(zoom, -zoom)
    ax[2].set_title(f"Ampliado: |u|, |v| ≤ {zoom}")

    for a in ax[1:]:
        a.set_xlabel("u (ciclos/px)")
        a.set_ylabel("v (ciclos/px)")
        a.axhline(0, color="w", lw=0.3, alpha=0.4)
        a.axvline(0, color="w", lw=0.3, alpha=0.4)

    plt.tight_layout()
    plt.show()

## 4. Elegir la imagen de entrada ✏️

Cambia `ENTRADA` por cualquier nombre del pool (basta con el número, p. ej. `"17"`),
o define `RUTA_ARCHIVO` para usar una imagen propia (se convierte a gris y se binariza
con umbral 128).

In [ ]:
ENTRADA = "05"          # nombre o número de una imagen del pool
RUTA_ARCHIVO = None     # p. ej. "imagenes/mi_imagen.png" para usar un archivo propio
VENTANA = False         # True para aplicar ventana de Hann
ZOOM = 0.125            # semiancho de la vista ampliada (ciclos/px)


def elegir(entrada, pool=POOL):
    if entrada in pool:
        return entrada, pool[entrada]
    coincidencias = [n for n in pool if n.startswith(f"{entrada}_") or n.startswith(f"{int(entrada):02d}_")] \
        if str(entrada).isdigit() else [n for n in pool if entrada in n]
    if not coincidencias:
        raise KeyError(f"'{entrada}' no está en el pool. Opciones: {list(pool)}")
    return coincidencias[0], pool[coincidencias[0]]


def cargar_archivo(ruta, umbral=128):
    import cv2
    img = cv2.imread(str(ruta), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(ruta)
    return (img >= umbral).astype(np.float64)


if RUTA_ARCHIVO:
    nombre, img = str(RUTA_ARCHIVO), cargar_archivo(RUTA_ARCHIVO)
else:
    nombre, img = elegir(ENTRADA)

analizar(img, nombre, ventana=VENTANA, zoom=ZOOM)

## 5. Comparaciones guiadas

Cada celda compara imágenes del pool en paralelo. Algunas preguntas para analizar:

1. **Perpendicularidad:** ¿qué orientación tiene el espectro de la vertical, la horizontal y la de 45°?
2. **Desplazamiento:** ¿cambia el espectro de *magnitud* al mover la recta (01 vs 03)? ¿Por qué? (pista: teorema del desplazamiento, solo cambia la fase).
3. **Linealidad:** ¿el espectro de la cruz se parece a la suma de los espectros de vertical + horizontal?
4. **Periodicidad:** en las familias de rectas, ¿a qué distancia del origen están los picos? Compara con $1/P$.
5. **Período vs. frecuencia:** al reducir el período a la mitad (12 vs 14), ¿qué pasa con la separación de los picos?
6. **Ángulos "incómodos":** ¿por qué la recta de 30° muestra líneas paralelas extra que la de 45° no?

In [ ]:
def comparar(nombres, ventana=False, zoom=None, pool=POOL, cmap="inferno"):
    # Una columna por imagen: arriba la imagen, abajo su espectro
    n = len(nombres)
    fig, ax = plt.subplots(2, n, figsize=(4.2 * n, 8.4))
    ax = np.atleast_2d(ax).reshape(2, n)
    for j, entrada in enumerate(nombres):
        nombre, img = elegir(entrada, pool)
        S, u, v = espectro(img, ventana)
        ax[0, j].imshow(img, vmin=0, vmax=1, interpolation="antialiased")
        ax[0, j].set_title(nombre, fontsize=10)
        ax[0, j].axis("off")
        ax[1, j].imshow(S, extent=[u[0], u[-1], v[-1], v[0]], cmap=cmap, interpolation="antialiased")
        if zoom:
            ax[1, j].set_xlim(-zoom, zoom)
            ax[1, j].set_ylim(zoom, -zoom)
        ax[1, j].set_xlabel("u")
        ax[1, j].set_ylabel("v")
    fig.suptitle("Imagen (arriba) y espectro log(1+|F|) (abajo)" + ("  [Hann]" if ventana else ""))
    plt.tight_layout()
    plt.show()

### 5.1 Una recta y rectas perpendiculares

In [ ]:
comparar(["01", "02", "03", "04"])

### 5.2 Diagonales y ángulos arbitrarios

In [ ]:
comparar(["05", "06", "07", "08", "09"])

**¿Por qué 30° y 60° muestran líneas paralelas extra?** Por *aliasing*. Una recta binaria
(bordes duros, escalonada) tiene un espectro que se extiende más allá de $\pm 0.5$
ciclos/px. Al muestrear, el espectro se replica cada 1 ciclo/px en $u$ y en $v$, y esas
réplicas de la recta caen dentro del cuadrado visible como **rectas paralelas** a la
principal. A 0°, 45° y 90° las réplicas coinciden con la propia recta, por eso no se notan.

Hay un segundo efecto, el *leakage*: a 30° la recta sale por bordes opuestos a distinta
altura y no empalma con su copia periódica. La ventana de Hann elimina ese efecto (se
limpia la textura fina), pero **no** las réplicas por aliasing:

In [ ]:
comparar(["05", "08", "09"], ventana=True)

### 5.3 Varias rectas por el origen

In [ ]:
comparar(["07", "10", "11"])

### 5.4 Familias de rectas paralelas

Vista ampliada para ver los picos. Con `PERIODO = 16`, la familia $(a, b)$ tiene picos en
$k\,(a, b)/16$: sobre el eje $u$ para las verticales, sobre la diagonal para $(1, 1)$ y en la
dirección $(1, 2)$ para la de pendiente 1:2. El espectro sigue siendo perpendicular a las rectas.

In [ ]:
comparar(["12", "13", "14", "15", "16"], zoom=0.25)

### 5.5 Rejillas

In [ ]:
comparar(["17", "18", "19"], zoom=0.25)

### 5.6 Formas cerradas

El cuadrado tiene lados horizontales y verticales: su espectro se concentra en los ejes.
El rombo (lados a ±45°) concentra la energía en las diagonales. El círculo no tiene una
dirección preferente: su espectro es radialmente simétrico.

In [ ]:
comparar(["20", "21", "22", "23"], zoom=0.25)

## 6. Verificación numérica de la periodicidad

Para la familia de rectas verticales, la posición de los picos a lo largo de $u$ debe ser
múltiplo de $1/\text{PERIODO}$.

In [ ]:
_, img = elegir("12")
F = np.abs(np.fft.fftshift(np.fft.fft2(img)))
u = np.fft.fftshift(np.fft.fftfreq(TAM))
fila = F[TAM // 2, :]

plt.figure(figsize=(12, 3.5))
plt.stem(u, fila, basefmt=" ")
for k in range(1, 8):
    plt.axvline(k / PERIODO, color="r", ls="--", lw=0.6)
    plt.axvline(-k / PERIODO, color="r", ls="--", lw=0.6)
plt.title(f"|F(u, 0)| de rectas verticales con período {PERIODO} px  (rojo: múltiplos de 1/{PERIODO})")
plt.xlabel("u (ciclos/px)")
plt.grid(alpha=0.3)
plt.show()

picos = u[fila > 0.01 * fila.max()]
print("Picos (ciclos/px):", np.round(picos[picos >= 0], 4))
print("Múltiplos de 1/P :", np.round(np.arange(0, 0.5, 1 / PERIODO), 4))